# Módulo 1 — Preparación de datos con Python

**Curso: Análisis y Pronóstico de Datos Mineros con Python**

> Tomar una base cruda de proceso y dejarla limpia, ordenada y con el tiempo bien interpretado.

---

### Cómo usar este notebook
1. Ábrelo en **Google Colab** y ejecuta las celdas **de arriba hacia abajo**.
2. Los datos se descargan solos desde el repositorio del curso; no tienes que subir nada.
3. Lee las salidas: cada bloque responde una pregunta concreta, no ejecutes por ejecutar.

In [ ]:
# Librerías base
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (12, 4)
pd.set_option('display.width', 120)

# --- Datos del curso -------------------------------------------------
# Los CSV viven en el repositorio del curso y se descargan solos.
# Si no hubiera internet, la función pide subir el archivo a mano.
REPO_DATOS = 'https://raw.githubusercontent.com/HishanFarfan/curso-datos-mineros/main/datos'

def cargar_datos(nombre, **kw):
    try:
        return pd.read_csv(f'{REPO_DATOS}/{nombre}', **kw)
    except Exception as e:
        print('No se pudo descargar desde GitHub:', e)
    try:
        from google.colab import files          # Colab: subir a mano
        print(f"Sube '{nombre}':")
        return pd.read_csv(next(iter(files.upload())), **kw)
    except ModuleNotFoundError:
        return pd.read_csv(nombre, **kw)         # local: archivo en el cwd

## 1. Carga de datos (versión de campo)

El archivo viene de un sistema en español: separador `;`, coma decimal y codificación `latin-1`. Si no se lo indicamos a pandas, las columnas numéricas se leerían como texto.

In [ ]:
df = cargar_datos('datos_proceso_planta.csv',
                  sep=';', decimal=',', encoding='latin-1')
df.head()

## 2. Primera inspección

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.dtypes

In [ ]:
# Encabezados con espacios sobrantes
list(df.columns)

In [ ]:
df.columns = df.columns.str.strip()
list(df.columns)

## 3. Resumen estadístico — buscar lo sospechoso

In [ ]:
df.describe()

El mínimo de `Tonelaje_tph` es `-999`: es un **código de 'sin medición'**, no un valor físico. Lo convertimos en dato ausente.

In [ ]:
df['Tonelaje_tph'] = df['Tonelaje_tph'].replace(-999, np.nan)
df['Tonelaje_tph'].describe()

## 4. El tiempo como variable

In [ ]:
df = df.reset_index(drop=True)
df['Fecha'] = pd.to_datetime(df['Fecha'], dayfirst=True)
df['Fecha'].head()

In [ ]:
# ¿Están ordenados cronológicamente?
df['Fecha'].is_monotonic_increasing

In [ ]:
df = df.sort_values('Fecha').reset_index(drop=True)
df['Fecha'].is_monotonic_increasing

## 5. Duplicados

In [ ]:
df.duplicated().sum()

In [ ]:
# Filas idénticas
df = df.drop_duplicates()
# Timestamps repetidos (dos lecturas para la misma hora)
df['Fecha'].duplicated().sum()

In [ ]:
# Criterio: promediar las lecturas de una misma hora
df = df.groupby('Fecha', as_index=False).agg({
    'Turno': 'first', 'Tipo_mineral': 'first',
    'Tonelaje_tph': 'mean', 'Ley_Cu_pct': 'mean',
    'Recuperacion_pct': 'mean', 'Potencia_kW': 'mean'})
df['Fecha'].duplicated().sum()

## 6. Índice temporal y frecuencia

In [ ]:
df = df.set_index('Fecha')
df = df.asfreq('h')   # impone paso horario; los huecos aparecen como NaN
df.shape

## 7. Valores faltantes

In [ ]:
df.isna().sum()

In [ ]:
(100 * df.isna().mean()).round(2)

### 7.1 Faltantes aislados vs. hueco largo

Interpolar una hora perdida es razonable. Reconstruir una **detención de planta** de 8 h sería inventar datos. Interpolamos solo huecos cortos.

In [ ]:
# Interpolación limitada a huecos de a lo más 2 horas
for col in ['Tonelaje_tph', 'Ley_Cu_pct', 'Recuperacion_pct', 'Potencia_kW']:
    df[col] = df[col].interpolate(method='time', limit=2, limit_area='inside')
df.isna().sum()

In [ ]:
# El sensor de potencia estuvo 'congelado' un tramo: valor repetido muchas horas
rep = df['Potencia_kW'].eq(df['Potencia_kW'].shift())
rep.groupby((~rep).cumsum()).sum().max()

## 8. Primera visualización

In [ ]:
df['Tonelaje_tph'].plot(title='Evolución del tonelaje')
plt.ylabel('t/h'); plt.show()

In [ ]:
df['Tonelaje_tph'].dropna().hist(bins=40)
plt.xlabel('t/h'); plt.title('Distribución del tonelaje'); plt.show()

In [ ]:
plt.boxplot(df['Tonelaje_tph'].dropna(), vert=True)
plt.ylabel('t/h'); plt.title('Boxplot tonelaje'); plt.show()

In [ ]:
plt.scatter(df['Tonelaje_tph'], df['Potencia_kW'], s=4, alpha=0.3)
plt.xlabel('Tonelaje [t/h]'); plt.ylabel('Potencia [kW]')
plt.title('Tonelaje vs Potencia'); plt.show()

## 9. Guardar la base preparada

In [ ]:
df.to_csv('serie_preparada_M1.csv')
df.describe()

## Actividades sugeridas

1. Repite la auditoría sobre `Ley_Cu_pct` y `Recuperacion_pct`.
2. ¿Cuántas horas de datos faltan en total y cómo se distribuyen en el tiempo?
3. Compara el histograma del tonelaje antes y después de reemplazar los `-999`.
4. Documenta en una celda de texto TODAS las decisiones de limpieza y su justificación.

---
## Cierre

La base quedó cargada, ordenada, con el tiempo como índice, la frecuencia regularizada y los problemas de calidad reconocidos. En el Módulo 2 cambiamos la pregunta: ¿qué información estadística podemos extraer?